## XXX model

$$
            H = J \sum_{i} \left( S_i^x S_{i+1}^x + S_i^y S_{i+1}^y + S_i^z S_{i+1}^z \right)
$$

In [1]:
import quante as qt

L = 10
mat = qt.generate.matrix.heisenberg_matrix(L, cyclic=True)
mat

array([[2.5, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1.5, 0.5, ..., 0. , 0. , 0. ],
       [0. , 0.5, 1.5, ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 1.5, 0.5, 0. ],
       [0. , 0. , 0. , ..., 0.5, 1.5, 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 2.5]])

In [2]:
builder = qt.generate.operas.SpinBuilder()
for i in range(L):
    builder += 'xx', [i, (i+1) % L], 1.0
    builder += 'yy', [i, (i+1) % L], 1.0
    builder += 'zz', [i, (i+1) % L], 1.0
ham = builder.build()
basis = qt.generate.basis.spin_basis(L)
ham.to_matrix(basis)

array([[2.5, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 1.5, 0.5, ..., 0. , 0. , 0. ],
       [0. , 0.5, 1.5, ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 1.5, 0.5, 0. ],
       [0. , 0. , 0. , ..., 0.5, 1.5, 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 2.5]])

In [3]:
import numpy as np
val, vec, _ = qt.linalg.krylov.eigsolve(mat, which='SR')
val

running Lanczos ...


array([-4.51544635])

In [4]:
qt.generate.solvable.heisenberg.xxx_finite_approx_ground_energy(L, j=1.)

np.float64(-4.516244912364189)

In [6]:
L = 20
mat = qt.generate.matrix.heisenberg_matrix(L, sparse=True, cyclic=True)
val, vec, _ = qt.linalg.krylov.eigsolve(mat, which='SR')
val

running Lanczos ...


array([-8.90438653])

In [7]:
qt.generate.solvable.heisenberg.xxx_finite_approx_ground_energy(L, j=1.)

np.float64(-8.904640565028531)

In [8]:
print(qt.generate.solvable.heisenberg.xxx_finite_approx_ground_energy(10000, j=1.)/10000)
print(qt.generate.solvable.heisenberg.xxx_infinite_ground_energy(j=1.))

-0.44314718878856313
-0.4431471805599453


## XX model

$$
    H = \sum_{i = 1}^{L - 1} J_i\left( S_i^x S_{i+1}^x + S_i^y S_{i+1}^y  \right) + \sum_{i = 1}^{L} h^z_i S^z_i
$$

In [9]:
import quante as qt
import numpy as np

L = 10
j = np.random.randn(L-1)
hz = np.random.randn(L)
mat = qt.generate.matrix.heisenberg_matrix(L, j=(j,j,0.), hz=hz)

In [11]:
val, vec, _ = qt.linalg.krylov.eigsolve(mat, which='SR')
val2 = qt.generate.solvable.heisenberg.xy_finite_ground_energy(
    L=L, jx=j, jy=j, jxy=0., jyx=0., hz=hz, pauli=False
)
val, val2

running Lanczos ...


(array([-4.06861167, -4.03625921]), np.float64(-4.068611670453926))

In [13]:
op = qt.generate.operas.spin
basis = qt.generate.basis.spin_basis(L)
state = qt.generate.state.neel(L, down_first=True)
tlist = np.linspace(0,10,100)
ns = [op.n(i).to_matrix(basis, sparse=True) for i in range(L)]
res1 = qt.linalg.evolve_and_measure(
    mat, state, tlist,
    measure=ns
)
res2 = np.real_if_close(list(qt.generate.solvable.heisenberg.xx_evolve(
    L=L, j=j, h=hz,
    init_state=['dn', 'up']*(L//2),
    tlist=tlist,
    obs_name='particle_number'
)))
np.allclose(res1, res2)

Evolving: 100%|##########| 100/100 [00:01<00:00, 90.62step/s, dt=0.10]


True

In [14]:
op = qt.generate.operas.spin
basis = qt.generate.basis.spin_basis(L)
state = qt.generate.state.neel(L, down_first=True)
tlist = [0,1]
def entanglement(t, state):
    return [qt.measure.entanglement_entropy(state.reshape(-1), left_number=i) for i in range(1,L)]
res1 = qt.linalg.evolve_and_measure(
    mat, state, tlist,
    measure=entanglement
)
res2 = np.real_if_close(list(qt.generate.solvable.heisenberg.xx_evolve(
    L=L, j=j, h=hz,
    init_state=['dn', 'up']*(L//2),
    tlist=tlist,
    obs_name='entanglement'
)))
np.allclose(res1, res2)

Evolving: 100%|##########| 2/2 [00:00<00:00, 181.86step/s, dt=1.00]


True

In [15]:
op = qt.generate.operas.spin
basis = qt.generate.basis.spin_basis(L)
state = qt.generate.state.neel(L, down_first=True)
tlist = [0,1]
def partial_dm(t, state):
    return qt.linalg.partial_trace(state.reshape(-1), dims=[2]*L, keep=[1,2])
res1 = qt.linalg.evolve_and_measure(
    mat, state, tlist,
    measure=partial_dm
)
print(res1[-1])
for s in qt.generate.solvable.heisenberg.xx_evolve(
        L=L, j=j, h=hz,
        init_state=['dn', 'up']*(L//2),
        tlist=tlist,
        obs_name='reduced_density_matrix',
        obs_para=([1,2], None)
    ):
    pass
s

100%|##########| 2/2 [00:00<00:00, 181.80it/s]


[[ 0.06114286+0.j          0.        +0.j          0.        +0.j          0.        +0.j        ]
 [ 0.        +0.j          0.13172081+0.j         -0.03412098-0.28525204j  0.        +0.j        ]
 [ 0.        +0.j         -0.03412098+0.28525204j  0.6838177 +0.j          0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j          0.12331863+0.j        ]]


Evolving: 100%|##########| 2/2 [00:00<00:00, 133.37step/s, dt=1.00]


array([[ 0.06114286+0.j        ,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.        +0.j        ,  0.13172081+0.j        , -0.03412098-0.28525204j,  0.        +0.j        ],
       [ 0.        +0.j        , -0.03412098+0.28525204j,  0.6838177 -0.j        ,  0.        +0.j        ],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ,  0.12331863+0.j        ]])

In [16]:
op = qt.generate.operas.spin
basis = qt.generate.basis.spin_basis(L)
state = qt.generate.state.neel(L, down_first=True)
tlist = [0,1]
def partial_dm(t, state):
    return qt.linalg.partial_trace(state.reshape(-1), dims=[2]*L, keep=[1,2])
res1 = qt.linalg.evolve_and_measure(
    mat, state, tlist,
    measure=partial_dm
)
print(res1[-1])
for s in qt.generate.solvable.heisenberg.xx_evolve(
        L=L, j=j, h=hz,
        init_state=['dn', 'up']*(L//2),
        tlist=tlist,
        obs_name='reduced_density_matrix',
        obs_para=([1,2], None)
    ):
    pass
s

100%|##########| 2/2 [00:00<00:00, 125.00it/s]


[[ 0.06114286+0.j          0.        +0.j          0.        +0.j          0.        +0.j        ]
 [ 0.        +0.j          0.13172081+0.j         -0.03412098-0.28525204j  0.        +0.j        ]
 [ 0.        +0.j         -0.03412098+0.28525204j  0.6838177 +0.j          0.        +0.j        ]
 [ 0.        +0.j          0.        +0.j          0.        +0.j          0.12331863+0.j        ]]


Evolving: 100%|##########| 2/2 [00:00<00:00, 117.66step/s, dt=1.00]


array([[ 0.06114286+0.j        ,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.        +0.j        ,  0.13172081+0.j        , -0.03412098-0.28525204j,  0.        +0.j        ],
       [ 0.        +0.j        , -0.03412098+0.28525204j,  0.6838177 -0.j        ,  0.        +0.j        ],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ,  0.12331863+0.j        ]])

## XY model

$$
            H = \sum_{i = 1}^{L - 1} \left(J^x_i S_i^x S_{i+1}^x + J^y_i S_i^y S_{i+1}^y + J^{xy}_i S_i^x S_{i+1}^y + J^{yx}_i S_i^y S_{i+ 1}^x \right) + \sum_{i = 1}^{L} h^z_i S^z_i
$$

In [17]:
import quante as qt
import numpy as np

L = 10
jx = jy = jxy = jyx = hz = 1
jx = np.random.randn(L-1)
# jx = jy
jxy = np.random.randn(L-1)
jyx = np.random.randn(L-1)
hz = np.random.randn(L)
mat = qt.generate.matrix.heisenberg_matrix(L, j=(jx,jy,0.), jxy=jxy, jyx=jyx, hz=hz)

In [18]:
val, vec, _ = qt.linalg.krylov.eigsolve(mat, which='SR')
val2 = qt.generate.solvable.heisenberg.xy_finite_ground_energy(
    L=L, jx=jx, jy=jy, jxy=jxy, jyx=jyx, hz=hz, pauli=False
)
val, val2

running Lanczos ...


(array([-5.3054283 , -5.29660119, -5.26193759, -5.25311048]),
 np.float64(-5.305428302099296))

In [19]:
val1 = np.linalg.eigvalsh(mat)
val2 = qt.generate.solvable.heisenberg.xy_spectrum(
    L=L, jx=jx, jy=jy, jxy=jxy, jyx=jyx, hz=hz, pauli=False
)
val1, val2

(array([-5.3054283 , -5.29660119, -5.26193759, ...,  5.26193759,  5.29660119,  5.3054283 ]),
 array([-5.3054283 , -5.29660119, -5.26193759, ...,  5.26193759,  5.29660119,  5.3054283 ]))

In [21]:
mat = qt.generate.matrix.heisenberg_matrix(L, j=(jx,jy,0.), jxy=jxy, jyx=jyx, hz=hz)
basis = qt.generate.basis.spin_basis(L)
state = qt.generate.state.neel(L, down_first=True)
tlist = np.linspace(0,10,100)
op = qt.generate.operas.spin
ns = [op.n(i).to_matrix(basis, sparse=True) for i in range(L)]
res1 = qt.linalg.evolve_and_measure(
    mat, state, tlist,
    measure=ns
)
res2 = np.real_if_close(list(qt.generate.solvable.heisenberg.xy_evolve(
    L=L, jx=jx, jy=jy, jxy=jxy, jyx=jyx, hz=hz,
    init_state=['dn', 'up']*(L//2) + ['dn']*(L%2),
    tlist=tlist,
    obs_name='particle_number'
)))
print(res1[-1])
print(res2[-1])


100%|##########| 100/100 [00:00<00:00, 170.97it/s]


[0.21512625 0.82874716 0.05680562 0.80375266 0.22799454 0.51960384 0.20466088 0.34640881 0.64566627 0.62173649]
[0.21512625 0.82874716 0.05680562 0.80375266 0.22799454 0.51960384 0.20466088 0.34640881 0.64566627 0.62173649]
